## Example: obtaining TSS-m6A-site pairs from CROWN-seq BAM files

### 1. Get conversion state of all A nucleotides in all 5' isoforms

Note: use -p to perform multiple processing to accelerate computation.

Output columns:

1. Chr
2. TSS (as in the read)
3. Strand
4. The nucleotide type of the TSS
5. The coverage of the TSS
6. The A site (Target)
7. A coverage
8. T coverage
9. C coverage
10. G coverage
11. N coverage
12. The distance to 5' (this has been adjusted by the splicing events)
13. The exon number of the A site locate in (0 - first exon, 1 - 2nd exon, etc.), although the column name is "EJC"


In [1]:
!python CROWN_seq_fetch_all_A_events_multiprocessing_v2.py -p 4 -r GRCh38.fa -b Jurkat_rep1.downsampled.bam -o Jurkat_rep1.TSS_m6A.txt

[2026-04-12 15:44:51] Running...
[2026-04-12 16:01:29] Merging TEMPs...
[2026-04-12 16:01:29] Genome pileup finished.


In [2]:
!head Jurkat_rep1.TSS_m6A.txt

10	134370	+	A	1	134370	1	0	0	0	0	0	0
10	134370	+	A	1	134373	0	0	0	1	0	3	0
10	134370	+	A	1	134374	0	0	0	1	0	4	0
10	134370	+	A	1	134375	0	0	0	1	0	5	0
10	134370	+	A	1	134378	0	0	0	1	0	8	0
10	134370	+	A	1	134384	0	0	0	1	0	14	0
10	134370	+	A	1	134391	0	0	0	1	0	21	0
10	134370	+	A	1	134392	0	0	0	1	0	22	0
10	134370	+	A	1	134401	0	0	0	1	0	31	0
10	134370	+	A	1	134402	0	0	0	1	0	32	0


### 2. For each 5' isoform, annotate the TSS

Notably, we cannot directly calculate the mapped coordinate of the TSS and the A sites, because a splicing event might break it. This step is trying to fix this problem and find the TSS for each A site.

In the output `Jurkat.TSS_exon_end.pairs.bed` file:

Columns:

1. Chr
2. The position of read end (0-based)
3. The position of read end (1-based)
4. The position of the TSS.
5. (Skipped)
6. Strand

In [3]:
# It splits the jobs based on chromosome id.
!python CROWN_fetch_first_exon.py -i Jurkat_rep1.TSS_m6A.txt -b Jurkat_rep1.bam -o Jurkat_rep1.TSS_exon_end.pairs.bed

In [4]:
!head Jurkat_rep1.TSS_exon_end.pairs.bed

10	931674	931675	931704	.	-
10	988518	988519	988434	.	+
10	988521	988522	988437	.	+
10	988526	988527	988452	.	+
10	988526	988527	988462	.	+
10	988526	988527	988464	.	+
10	988526	988527	988466	.	+
10	1048854	1048855	1048886	.	-
10	1048857	1048858	1048889	.	-
10	1048857	1048858	1048890	.	-


In [5]:
# This is in important intermediate file from CROWN_fetch_first_exon.py. We use A's >=20 reads to save space and calculation. Change the threshold in the script file.
!Jurkat_rep1.TSS_m6A.txt.all.cov20.csv

/bin/bash: Jurkat_rep1.TSS_m6A.txt.all.cov20.csv: command not found


### 3. Run BEDTools to find the closest first exon end for each TSS.

If the read 

In [6]:
!bedtools sort -i Jurkat_rep1.TSS_exon_end.pairs.bed  > Jurkat_rep1.TSS_exon_end.pairs.sorted.bed

In [7]:
!bedtools closest -s -D a -a Jurkat_rep1.TSS_exon_end.pairs.sorted.bed -b gencode.v45.primary_assembly.annotation.nochr.anno.first_exon_end.bed > Jurkat_rep1.TSS_exon_end_pairs.closest.bed

### 4. Annotate the 3' exon ends.

If the reads of the 5' isoform get spliced, the splicing site is used. If the read is not spliced, the exon end annotation (in step 2) will be used to represent the 3' exon end.

In [10]:
!python get_TSS_m6A_pair_to_first_exon_end.py -i Jurkat_rep1.TSS_m6A.txt.all.cov20.csv -b Jurkat_rep1.TSS_exon_end_pairs.closest.bed -o Jurkat_rep1.TSS_m6A.txt.all.cov20.annot_exon.csv

In [11]:
!head Jurkat_rep1.TSS_m6A.txt.all.cov20.annot_exon.csv

,chr,TSS,strand,TSS_base,TSS_cov,Target,A,T,C,G,N,dist_to_5p,EJC,cov,level,first_exon_end_original,first_exon_end_fixed,m6A_to_exon_end_original,m6A_to_exon_end_fixed
349,10,931704,-,A,22,931704,22,0,0,0,0,0,0,22,1.0,931675,931428,29,276
350,10,931704,-,A,22,931690,0,0,0,22,0,14,0,22,0.0,931675,931428,15,262
351,10,931704,-,A,22,931685,2,0,0,20,0,19,0,22,0.0909090909090909,931675,931428,10,257
352,10,931704,-,A,22,931684,0,0,0,22,0,20,0,22,0.0,931675,931428,9,256
353,10,931704,-,A,22,931679,2,0,0,20,0,25,0,22,0.0909090909090909,931675,931428,4,251
354,10,931704,-,A,22,931678,0,0,0,22,0,26,0,22,0.0,931675,931428,3,250
416,10,988434,+,G,29,988434,0,0,0,29,0,0,0,29,0.0,988519,988527,85,93
417,10,988434,+,G,29,988436,0,0,0,29,0,2,0,29,0.0,988519,988527,83,91
418,10,988434,+,G,29,988437,0,0,0,29,0,3,0,29,0.0,988519,988527,82,90
